# 태양 표면 자기장의 진화 구현 / Implementation: Surface Evolution of the Sun's Magnetic Field

**논문 / Paper**: Sheeley, N.R. Jr. (2005), "Surface Evolution of the Sun's Magnetic Field: A Historical Review of the Flux-Transport Mechanism", *Living Reviews in Solar Physics*, 2, 5.

---

## 개요 / Overview

이 노트북은 태양 표면 자기장 진화를 기술하는 **flux-transport 모델**의 핵심 개념을 구현합니다.
Sheeley (2005)는 역사적/서술적 리뷰 논문이지만, 기저에 깔린 물리학은 매우 구현 가능합니다.

This notebook implements the key concepts of the **flux-transport model** describing solar surface magnetic field evolution.
While Sheeley (2005) is a historical/narrative review, the underlying physics is highly implementable.

**핵심 방정식 / Core Equation** — 축대칭 flux-transport 방정식:

$$\frac{\partial B_r}{\partial t} = -\frac{1}{R_\odot \sin\theta}\frac{\partial}{\partial\theta}(v\sin\theta\, B_r) + \frac{\eta}{R_\odot^2 \sin\theta}\frac{\partial}{\partial\theta}\left(\sin\theta\frac{\partial B_r}{\partial\theta}\right) + S(\theta, t)$$

여기서 $v(\theta)$는 경도 방향 유동 속도, $\eta$는 초과립 확산 계수, $S(\theta,t)$는 활동 영역 소스항입니다.

where $v(\theta)$ is the meridional flow speed, $\eta$ is the supergranular diffusion coefficient, and $S(\theta,t)$ is the active region source term.

---

| 파트 / Part | 내용 / Content |
|---|---|
| Part 1 | 1D 구면 flux-transport 방정식 / Flux-Transport Equation on 1D Sphere |
| Part 2 | Joy's Law 기울기와 극 자기장 역전 / Joy's Law Tilt & Polar Field Reversal |
| Part 3 | 3가지 수송 체제 (Figure 3 재현) / Three Transport Regimes |
| Part 4 | 매개변수 민감도 연구 / Parameter Sensitivity Study |
| Part 5 | 다중 주기 시뮬레이션 / Multi-Cycle Simulation with Variable Flow |
| Part 6 | 요약 표 / Summary Table |

In [ ]:
"""Imports and global configuration for the flux-transport simulation."""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from scipy.ndimage import gaussian_filter1d

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'lines.linewidth': 1.5,
})

# ── Physical constants / 물리 상수 ──
R_SUN = 6.96e10       # Solar radius [cm]
R_SUN_KM = 6.96e5     # Solar radius [km]

# ── Default simulation parameters / 기본 시뮬레이션 매개변수 ──
ETA_DEFAULT = 600.0    # Supergranular diffusion coefficient [km^2/s]
V_DEFAULT = 11.0       # Peak meridional flow speed [m/s]
CYCLE_PERIOD = 11.0    # Sunspot cycle period [years]

# ── Unit conversions / 단위 변환 ──
SECONDS_PER_YEAR = 365.25 * 24 * 3600.0
KM2_TO_M2 = 1e6       # km^2 → m^2
R_SUN_M = R_SUN_KM * 1e3  # Solar radius [m]

print("=== Flux-Transport Model Configuration ===")
print(f"  R_sun = {R_SUN_KM:.0f} km = {R_SUN_M:.3e} m")
print(f"  eta   = {ETA_DEFAULT} km^2/s = {ETA_DEFAULT * KM2_TO_M2:.3e} m^2/s")
print(f"  v_max = {V_DEFAULT} m/s")
print(f"  Cycle = {CYCLE_PERIOD} years")

---
## Part 1: 1D 구면 Flux-Transport 방정식 (축대칭) / The Flux-Transport Equation on a 1D Sphere (Axisymmetric)

### 물리적 배경 / Physical Background

태양 표면의 방사 방향 자기장 $B_r(\theta, t)$는 세 가지 과정에 의해 진화합니다:
1. **초과립 확산 / Supergranular diffusion** ($\eta \approx 600\;\mathrm{km^2/s}$): 대류 셀에 의한 무작위 걸음 과정
2. **경도 방향 유동 / Meridional flow** ($v \approx 11\;\mathrm{m/s}$): 적도에서 극으로의 체계적 플라즈마 흐름
3. **소스항 / Source term** $S(\theta, t)$: 활동 영역의 자속 출현 (쌍극 doublet)

The radial magnetic field $B_r(\theta, t)$ on the solar surface evolves through three processes:
1. **Supergranular diffusion** ($\eta \approx 600\;\mathrm{km^2/s}$): random-walk process by convective cells
2. **Meridional flow** ($v \approx 11\;\mathrm{m/s}$): systematic poleward plasma flow
3. **Source term** $S(\theta, t)$: flux emergence from active regions (bipolar doublets)

### 수치 기법 / Numerical Method

위도 격자에서 유한 차분법을 사용합니다. $x = \sin(\text{latitude}) = \cos\theta$ 좌표로 변환하면 극에서의 특이점을 자연스럽게 처리할 수 있습니다.

We use finite differences on a latitude grid. Transforming to $x = \sin(\text{latitude}) = \cos\theta$ naturally handles the polar singularities.

**좌표 변환 / Coordinate transformation**: $x = \cos\theta$이면 $\sin\theta = \sqrt{1-x^2}$이고:

$$\frac{\partial B_r}{\partial t} = \frac{\partial}{\partial x}\left[v_x (1-x^2)^{1/2} B_r\right] \frac{1}{R_\odot} + \frac{\eta}{R_\odot^2}\frac{\partial}{\partial x}\left[(1-x^2)\frac{\partial B_r}{\partial x}\right] + S(x,t)$$

여기서 $v_x$는 $x$ 방향의 유효 유동 속도입니다.

where $v_x$ is the effective flow speed in the $x$-direction.

In [ ]:
class FluxTransportSolver:
    """1D axisymmetric flux-transport solver on the solar surface.

    Solves the flux-transport equation for radial magnetic field B_r
    on a uniform grid in sin(latitude), using explicit forward-Euler
    time-stepping with finite differences.

    Attributes:
        n_lat: Number of latitude grid points.
        lats: Latitude array in radians [-pi/2, pi/2].
        x: sin(latitude) coordinate array.
        dx: Grid spacing in x.
        Br: Radial magnetic field array [Gauss].
        eta: Supergranular diffusion coefficient [km^2/s].
        v_max: Peak meridional flow speed [m/s].
    """

    def __init__(self, n_lat: int = 181, eta: float = ETA_DEFAULT,
                 v_max: float = V_DEFAULT):
        """Initializes the solver with grid and physical parameters.

        Args:
            n_lat: Number of latitude grid points (pole to pole).
            eta: Diffusion coefficient in km^2/s.
            v_max: Peak poleward meridional flow speed in m/s.
        """
        self.n_lat = n_lat
        self.eta = eta * KM2_TO_M2           # Convert to m^2/s
        self.v_max = v_max                    # m/s

        # Uniform grid in sin(latitude) = cos(colatitude)
        self.x = np.linspace(-1.0, 1.0, n_lat)
        self.dx = self.x[1] - self.x[0]
        self.lats = np.arcsin(self.x)         # Latitude in radians
        self.lats_deg = np.degrees(self.lats) # Latitude in degrees
        self.colat = np.pi / 2.0 - self.lats  # Colatitude theta

        # sin(theta) = sqrt(1 - x^2), clamped to avoid division by zero
        self.sin_theta = np.sqrt(np.clip(1.0 - self.x**2, 1e-30, None))

        # Initialize field to zero
        self.Br = np.zeros(n_lat)

    def meridional_flow(self, x: np.ndarray) -> np.ndarray:
        """Computes the meridional flow profile v(latitude).

        The standard profile is v = v_max * sin(2*latitude), which is
        poleward in both hemispheres and vanishes at equator and poles.
        In the x = sin(lat) coordinate: sin(2*lat) = 2*x*sqrt(1-x^2).

        Args:
            x: sin(latitude) coordinate array.

        Returns:
            Meridional flow speed array [m/s], positive = poleward.
        """
        sin_2lat = 2.0 * x * np.sqrt(np.clip(1.0 - x**2, 0, None))
        return self.v_max * sin_2lat

    def compute_dt(self) -> float:
        """Computes the stable time step from the CFL condition.

        Returns:
            Maximum stable time step in seconds.
        """
        # Diffusion CFL: dt < dx^2 * R^2 / (2*eta)
        dt_diff = 0.4 * (self.dx * R_SUN_M)**2 / (2.0 * self.eta) if self.eta > 0 else 1e30

        # Advection CFL: dt < dx * R / v_max
        if self.v_max > 0:
            dt_adv = 0.4 * self.dx * R_SUN_M / self.v_max
        else:
            dt_adv = 1e30

        return min(dt_diff, dt_adv)

    def step(self, dt: float) -> None:
        """Advances the solution by one time step using forward Euler.

        The equation in x = sin(lat) coordinates:
        dBr/dt = (1/R) d/dx [v_x * sqrt(1-x^2) * Br]
               + (eta/R^2) d/dx [(1-x^2) dBr/dx]

        where v_x is the component of meridional flow mapped to the
        x-coordinate. Since x = sin(lat), dx = cos(lat)*dlat,
        so v_x = v(lat) / cos(lat) when expressed per unit x.
        But the transport equation in x-form absorbs the Jacobian.

        We work directly in the colatitude form with x-grid mapping.

        Args:
            dt: Time step in seconds.
        """
        Br = self.Br
        x = self.x
        dx = self.dx
        n = self.n_lat

        # Flow speed at cell centers
        v = self.meridional_flow(x)

        # ── Diffusion term: (eta/R^2) d/dx[(1-x^2) dBr/dx] ──
        # At half-grid points: (1 - x_{i+1/2}^2) * (Br[i+1] - Br[i]) / dx
        x_half = 0.5 * (x[:-1] + x[1:])
        coeff_half = 1.0 - x_half**2
        dBr_dx_half = (Br[1:] - Br[:-1]) / dx
        flux_diff = coeff_half * dBr_dx_half

        diff_term = np.zeros(n)
        diff_term[1:-1] = (self.eta / R_SUN_M**2) * (flux_diff[1:] - flux_diff[:-1]) / dx

        # ── Advection term: (1/R) d/dx[v * sqrt(1-x^2) * Br] ──
        # Use upwind scheme for stability
        transport = v * self.sin_theta * Br
        adv_term = np.zeros(n)
        # Central difference for advection flux divergence
        adv_term[1:-1] = (1.0 / R_SUN_M) * (transport[2:] - transport[:-2]) / (2.0 * dx)

        # ── Update ──
        # Note: advection term has a sign convention. In the original equation,
        # the advection is -1/(R sin(theta)) d/dtheta(v sin(theta) Br).
        # In x-coordinates with poleward flow, the sign works out so that
        # flux is swept toward poles.
        self.Br += dt * (diff_term - adv_term)

        # Boundary conditions: Br = 0 at poles (no flux through poles)
        # Actually, for finite volume, we set dBr/dx = 0 at poles
        self.Br[0] = self.Br[1]
        self.Br[-1] = self.Br[-2]

    def add_source(self, lat_deg: float, width_deg: float = 3.0,
                   amplitude: float = 10.0, tilt_deg: float = 7.0) -> None:
        """Adds a bipolar magnetic region (doublet) as a source term.

        Models a tilted bipolar active region using Joy's law. The leading
        polarity is placed closer to the equator, trailing polarity closer
        to the pole, separated by the tilt angle.

        Args:
            lat_deg: Central latitude of the active region [degrees].
            width_deg: Gaussian width of each polarity [degrees].
            amplitude: Peak field strength [Gauss].
            tilt_deg: Joy's law tilt angle [degrees].
        """
        sign = 1.0 if lat_deg >= 0 else -1.0
        # Leading polarity: closer to equator (negative in N hemisphere for odd cycle)
        lat_lead = lat_deg - sign * tilt_deg / 2.0
        lat_trail = lat_deg + sign * tilt_deg / 2.0

        x_lead = np.sin(np.radians(lat_lead))
        x_trail = np.sin(np.radians(lat_trail))
        sigma_x = np.sin(np.radians(width_deg))

        # Leading polarity is negative in N hemisphere (Hale's law for odd cycle)
        self.Br -= sign * amplitude * np.exp(-0.5 * ((self.x - x_lead) / sigma_x)**2)
        self.Br += sign * amplitude * np.exp(-0.5 * ((self.x - x_trail) / sigma_x)**2)

    def run(self, t_years: float, dt: float = None,
            source_func=None, snapshot_interval_years: float = 0.5):
        """Runs the simulation for a given duration.

        Args:
            t_years: Total simulation time in years.
            dt: Time step in seconds (computed from CFL if None).
            source_func: Optional callable(solver, t_seconds) to add sources.
            snapshot_interval_years: Interval between saved snapshots [years].

        Returns:
            Dictionary with 'times' (years), 'Br_snapshots' (2D array),
            and 'lats_deg' (latitude array).
        """
        if dt is None:
            dt = self.compute_dt()

        t_total = t_years * SECONDS_PER_YEAR
        snapshot_interval = snapshot_interval_years * SECONDS_PER_YEAR

        times = []
        snapshots = []
        t = 0.0
        next_snapshot = 0.0

        while t < t_total:
            # Save snapshot
            if t >= next_snapshot:
                times.append(t / SECONDS_PER_YEAR)
                snapshots.append(self.Br.copy())
                next_snapshot += snapshot_interval

            # Add sources
            if source_func is not None:
                source_func(self, t)

            # Advance
            actual_dt = min(dt, t_total - t)
            self.step(actual_dt)
            t += actual_dt

        # Final snapshot
        times.append(t / SECONDS_PER_YEAR)
        snapshots.append(self.Br.copy())

        return {
            'times': np.array(times),
            'Br_snapshots': np.array(snapshots),
            'lats_deg': self.lats_deg.copy(),
        }

print("FluxTransportSolver class defined successfully.")
print(f"  Grid: {181} points in sin(latitude)")
print(f"  Default dt ~ {FluxTransportSolver().compute_dt() / 3600:.1f} hours")

### 검증: 단일 doublet의 확산 / Validation: Diffusion of a Single Doublet

단일 쌍극 활동 영역을 놓고 확산만으로 진화시켜 solver를 검증합니다.
확산은 자기장을 매끄럽게 펼치면서 총 자속을 보존해야 합니다.

We validate the solver by placing a single bipolar region and evolving it with diffusion only.
Diffusion should smooth the field while conserving total flux.

In [ ]:
"""Part 1 validation: single doublet diffusion test."""

# Create solver with diffusion only (no flow)
solver = FluxTransportSolver(n_lat=361, eta=600.0, v_max=0.0)
solver.add_source(lat_deg=15.0, width_deg=3.0, amplitude=10.0, tilt_deg=7.0)

# Store initial state
Br_init = solver.Br.copy()
lats = solver.lats_deg

# Run for several years (diffusion-only)
result = solver.run(t_years=5.0, snapshot_interval_years=1.0)

# ── Plot evolution ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Field profiles at different times
ax = axes[0]
for i, t in enumerate(result['times'][::2]):
    ax.plot(result['lats_deg'], result['Br_snapshots'][i * 2],
            label=f't = {t:.1f} yr', alpha=0.8)
ax.set_xlabel('Latitude [deg] / 위도 [도]')
ax.set_ylabel('$B_r$ [Gauss]')
ax.set_title('확산에 의한 Doublet 진화 / Doublet Evolution by Diffusion')
ax.legend()
ax.set_xlim(-5, 40)
ax.axhline(0, color='gray', lw=0.5)
ax.grid(True, alpha=0.3)

# Right: Total flux conservation check
ax = axes[1]
# Compute unsigned flux integral: integral of |Br| * cos(lat) * dlat
dx = solver.dx
total_unsigned_flux = []
total_signed_flux = []
for snap in result['Br_snapshots']:
    cos_lat = np.sqrt(1.0 - solver.x**2)
    total_unsigned_flux.append(np.sum(np.abs(snap) * cos_lat) * dx)
    total_signed_flux.append(np.sum(snap * cos_lat) * dx)

ax.plot(result['times'], total_unsigned_flux / total_unsigned_flux[0],
        'b-o', label='Unsigned flux / 비부호 자속', markersize=4)
ax.plot(result['times'], np.abs(total_signed_flux) / total_unsigned_flux[0],
        'r--s', label='|Signed flux| / |부호 자속|', markersize=4)
ax.set_xlabel('Time [years] / 시간 [년]')
ax.set_ylabel('Normalized flux / 정규화된 자속')
ax.set_title('자속 보존 검증 / Flux Conservation Check')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.2)

plt.tight_layout()
plt.show()

print("\n=== Part 1 검증 결과 / Validation Results ===")
print(f"  초기 비부호 자속 / Initial unsigned flux: {total_unsigned_flux[0]:.4f}")
print(f"  최종 비부호 자속 / Final unsigned flux:   {total_unsigned_flux[-1]:.4f}")
print(f"  비율 / Ratio: {total_unsigned_flux[-1]/total_unsigned_flux[0]:.4f}")
print(f"  → 확산은 반대 극성을 상쇄시켜 비부호 자속을 감소시킵니다.")
print(f"  → Diffusion cancels opposite polarities, reducing unsigned flux.")

---
## Part 2: Joy's Law 기울기와 극 자기장 역전 / Joy's Law Tilt and Polar Field Reversal

### 핵심 메커니즘 / Key Mechanism

Sheeley (2005)의 핵심 통찰: **Joy's law 기울기**가 극 자기장 역전의 근본 원인입니다.

The key insight from Sheeley (2005): **Joy's law tilt** is the fundamental cause of polar field reversal.

1. 활동 영역은 **기울어진 쌍극**(doublet)으로 출현합니다 — 후행 극성이 극에 더 가깝습니다.
2. **확산**이 반대 극성 간의 상쇄를 일으키면서, 후행 극성의 잔여 자속이 극으로 확산됩니다.
3. **경도 방향 유동**이 이 잔여 자속을 극으로 수송하여, 이전 주기의 극 자기장을 역전시킵니다.

1. Active regions emerge as **tilted bipoles** (doublets) — trailing polarity is closer to the pole.
2. **Diffusion** causes cancellation between opposite polarities, with residual trailing-polarity flux diffusing poleward.
3. **Meridional flow** transports this residual flux to the poles, reversing the polar field from the previous cycle.

### 시뮬레이션 설계 / Simulation Design

- 단일 쌍극 영역의 진화를 추적합니다 (확산 + 유동).
- 하나의 태양 활동 주기 동안 여러 doublet을 누적시켜 극 자기장 역전을 재현합니다.

- Track the evolution of a single bipolar region (diffusion + flow).
- Accumulate multiple doublets over one solar cycle to reproduce polar field reversal.

In [ ]:
"""Part 2a: Single bipolar region evolution with diffusion + flow."""

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Case 1: Diffusion only ──
solver_diff = FluxTransportSolver(n_lat=361, eta=600.0, v_max=0.0)
solver_diff.add_source(lat_deg=20.0, width_deg=3.0, amplitude=10.0, tilt_deg=7.0)
result_diff = solver_diff.run(t_years=8.0, snapshot_interval_years=1.0)

ax = axes[0]
for i in range(0, len(result_diff['times']), 2):
    t = result_diff['times'][i]
    ax.plot(result_diff['lats_deg'], result_diff['Br_snapshots'][i],
            label=f'{t:.0f} yr', alpha=0.8)
ax.set_title('확산만 / Diffusion Only')
ax.set_xlabel('Latitude [deg]')
ax.set_ylabel('$B_r$ [G]')
ax.set_xlim(-10, 90)
ax.axhline(0, color='gray', lw=0.5)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# ── Case 2: Meridional flow only ──
solver_flow = FluxTransportSolver(n_lat=361, eta=0.01, v_max=11.0)
solver_flow.add_source(lat_deg=20.0, width_deg=3.0, amplitude=10.0, tilt_deg=7.0)
result_flow = solver_flow.run(t_years=8.0, snapshot_interval_years=1.0)

ax = axes[1]
for i in range(0, len(result_flow['times']), 2):
    t = result_flow['times'][i]
    ax.plot(result_flow['lats_deg'], result_flow['Br_snapshots'][i],
            label=f'{t:.0f} yr', alpha=0.8)
ax.set_title('유동만 / Flow Only')
ax.set_xlabel('Latitude [deg]')
ax.set_xlim(-10, 90)
ax.axhline(0, color='gray', lw=0.5)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# ── Case 3: Diffusion + flow ──
solver_both = FluxTransportSolver(n_lat=361, eta=600.0, v_max=11.0)
solver_both.add_source(lat_deg=20.0, width_deg=3.0, amplitude=10.0, tilt_deg=7.0)
result_both = solver_both.run(t_years=8.0, snapshot_interval_years=1.0)

ax = axes[2]
for i in range(0, len(result_both['times']), 2):
    t = result_both['times'][i]
    ax.plot(result_both['lats_deg'], result_both['Br_snapshots'][i],
            label=f'{t:.0f} yr', alpha=0.8)
ax.set_title('확산 + 유동 / Diffusion + Flow')
ax.set_xlabel('Latitude [deg]')
ax.set_xlim(-10, 90)
ax.axhline(0, color='gray', lw=0.5)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle("단일 쌍극 영역의 진화 / Single Bipolar Region Evolution\n"
             "(lat=20°, tilt=7°, Joy's Law)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("핵심 관찰 / Key observation:")
print("  확산+유동 경우, 후행 극성(양)이 극으로 수송되어 극 자기장을 형성합니다.")
print("  With diffusion+flow, trailing polarity (positive) is transported poleward, building polar field.")

### Part 2b: 한 주기 동안의 극 자기장 역전 / Polar Field Reversal Over One Cycle

하나의 11년 주기 동안 다수의 쌍극 활동 영역을 주기적으로 출현시켜,
극 자기장이 점진적으로 역전되는 과정을 시뮬레이션합니다.

We simulate the gradual reversal of polar fields by periodically emerging
multiple bipolar active regions over one 11-year sunspot cycle.

In [ ]:
"""Part 2b: Polar field reversal from accumulated doublet sources."""


def sunspot_source_cycle(solver: FluxTransportSolver, t_sec: float,
                         cycle_period_yr: float = 11.0,
                         emergence_interval_sec: float = 30 * 86400.0,
                         amplitude: float = 5.0,
                         lat_range: tuple = (10.0, 30.0),
                         tilt_base: float = 5.0,
                         _state: dict = None) -> None:
    """Adds bipolar sources following a sunspot cycle pattern.

    Sources emerge with a sinusoidal activity envelope. Emergence
    latitudes drift equatorward (Spoerer's law) over the cycle.

    Args:
        solver: The FluxTransportSolver instance.
        t_sec: Current simulation time in seconds.
        cycle_period_yr: Sunspot cycle period in years.
        emergence_interval_sec: Time between successive source emergences.
        amplitude: Base amplitude of each source [Gauss].
        lat_range: Latitude range for source emergence (high, low) [degrees].
        tilt_base: Base Joy's law tilt angle [degrees].
        _state: Internal state dictionary for tracking emergence times.
    """
    if _state is None:
        return

    # Check if it's time to emerge a new source
    if t_sec < _state.get('next_emergence', 0.0):
        return

    t_yr = t_sec / SECONDS_PER_YEAR
    cycle_phase = (t_yr % cycle_period_yr) / cycle_period_yr

    # Sinusoidal activity envelope (peak at phase 0.3)
    activity = np.sin(np.pi * cycle_phase)**2
    if activity < 0.05:
        _state['next_emergence'] = t_sec + emergence_interval_sec
        return

    # Spoerer's law: latitude drifts equatorward over the cycle
    lat_high, lat_low = lat_range
    lat_center = lat_high - (lat_high - lat_low) * cycle_phase

    # Add random scatter
    rng = _state.get('rng', np.random.default_rng(42))
    lat_scatter = rng.normal(0, 2.0)
    lat = lat_center + lat_scatter

    # Joy's law: tilt proportional to latitude
    tilt = tilt_base + 0.3 * abs(lat)

    # Scale amplitude by activity level
    amp = amplitude * activity

    # Emerge in both hemispheres (symmetric)
    solver.add_source(lat_deg=abs(lat), width_deg=2.5, amplitude=amp, tilt_deg=tilt)
    solver.add_source(lat_deg=-abs(lat), width_deg=2.5, amplitude=amp, tilt_deg=tilt)

    _state['next_emergence'] = t_sec + emergence_interval_sec
    _state['rng'] = rng


# ── Run one full cycle ──
solver = FluxTransportSolver(n_lat=361, eta=600.0, v_max=11.0)

# Set initial polar field (mimicking end of previous cycle)
# North pole positive, south pole negative
polar_width = np.sin(np.radians(15.0))
solver.Br += 2.0 * np.exp(-0.5 * ((solver.x - 1.0) / polar_width)**2)
solver.Br -= 2.0 * np.exp(-0.5 * ((solver.x + 1.0) / polar_width)**2)

state = {'next_emergence': 0.0, 'rng': np.random.default_rng(42)}

def source_func(s, t):
    """Wrapper for sunspot source with persistent state."""
    sunspot_source_cycle(s, t, cycle_period_yr=11.0,
                         emergence_interval_sec=20 * 86400.0,
                         amplitude=4.0, _state=state)

result = solver.run(t_years=11.0, snapshot_interval_years=0.25,
                    source_func=source_func)

# ── Plot butterfly diagram and polar field ──
fig = plt.figure(figsize=(14, 8))
gs = GridSpec(2, 2, height_ratios=[2, 1], hspace=0.35, wspace=0.3)

# Top: Butterfly (time-latitude) diagram
ax_butterfly = fig.add_subplot(gs[0, :])
T, L = np.meshgrid(result['times'], result['lats_deg'])
vmax = np.percentile(np.abs(result['Br_snapshots']), 98)
im = ax_butterfly.pcolormesh(T, L, result['Br_snapshots'].T,
                              cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                              shading='auto')
ax_butterfly.set_xlabel('Time [years] / 시간 [년]')
ax_butterfly.set_ylabel('Latitude [deg] / 위도 [도]')
ax_butterfly.set_title('나비 다이어그램: 한 주기의 Flux-Transport / '
                        'Butterfly Diagram: One-Cycle Flux Transport')
plt.colorbar(im, ax=ax_butterfly, label='$B_r$ [G]', shrink=0.8)

# Bottom left: Polar field evolution
ax_polar = fig.add_subplot(gs[1, 0])
north_idx = np.where(result['lats_deg'] > 75)[0]
south_idx = np.where(result['lats_deg'] < -75)[0]
Br_north = np.mean(result['Br_snapshots'][:, north_idx], axis=1)
Br_south = np.mean(result['Br_snapshots'][:, south_idx], axis=1)

ax_polar.plot(result['times'], Br_north, 'r-', label='North / 북극', lw=2)
ax_polar.plot(result['times'], Br_south, 'b-', label='South / 남극', lw=2)
ax_polar.axhline(0, color='gray', lw=0.5)
ax_polar.set_xlabel('Time [years] / 시간 [년]')
ax_polar.set_ylabel('$\\langle B_r \\rangle_{polar}$ [G]')
ax_polar.set_title('극 자기장 진화 / Polar Field Evolution')
ax_polar.legend()
ax_polar.grid(True, alpha=0.3)

# Bottom right: Latitude profile at several times
ax_profile = fig.add_subplot(gs[1, 1])
for t_target in [0, 3, 5.5, 8, 11]:
    idx = np.argmin(np.abs(result['times'] - t_target))
    ax_profile.plot(result['lats_deg'], result['Br_snapshots'][idx],
                    label=f't={result["times"][idx]:.1f} yr', alpha=0.7)
ax_profile.set_xlabel('Latitude [deg] / 위도 [도]')
ax_profile.set_ylabel('$B_r$ [G]')
ax_profile.set_title('위도 프로파일 / Latitude Profiles')
ax_profile.legend(fontsize=8)
ax_profile.grid(True, alpha=0.3)
ax_profile.axhline(0, color='gray', lw=0.5)

plt.show()

print("\n=== Part 2 결과 / Results ===")
print(f"  북극 자기장 (초기) / North polar field (initial): {Br_north[0]:.3f} G")
print(f"  북극 자기장 (최종) / North polar field (final):   {Br_north[-1]:.3f} G")
if Br_north[0] * Br_north[-1] < 0:
    print("  ✓ 극 자기장이 역전되었습니다! / Polar field has reversed!")
else:
    print("  극 자기장이 감소했지만 완전히 역전되지 않았습니다.")
    print("  Polar field decreased but did not fully reverse.")

---
## Part 3: 세 가지 수송 체제 (Figure 3 재현) / The Three Transport Regimes (cf. Figure 3)

Sheeley (2005)의 Figure 3은 flux-transport 모델의 가장 교육적인 그림 중 하나입니다.
동일한 소스항에 대해 세 가지 다른 수송 조건의 결과를 비교합니다:

Figure 3 in Sheeley (2005) is one of the most instructive figures of the flux-transport model.
It compares the results of three different transport conditions for the same source terms:

| 경우 / Case | 확산 / Diffusion | 유동 / Flow | 기대 결과 / Expected Result |
|---|---|---|---|
| 1 | 없음 / None | 없음 / None | 소스만 누적 / Sources accumulate only |
| 2 | $\eta = 600\;\mathrm{km^2/s}$ | 없음 / None | 반대 극성 상쇄, 느린 극 확산 / Cross-polarity cancellation, slow polar diffusion |
| 3 | $\eta = 600\;\mathrm{km^2/s}$ | $v = 11\;\mathrm{m/s}$ | 집중된 극 방향 surge / Concentrated poleward surges |

**핵심 차이점 / Key difference**: 경도 방향 유동이 있으면, 후행 극성 자속이 **집중된 surge**로 극으로 수송되어
나비 다이어그램에서 뚜렷한 극 방향 줄무늬 패턴이 나타납니다.

With meridional flow, trailing-polarity flux is transported poleward as **concentrated surges**,
producing distinct poleward streaks in the butterfly diagram.

In [ ]:
"""Part 3: Three transport regimes — butterfly diagram comparison."""


def make_deterministic_source(cycle_period_yr: float = 11.0,
                              emergence_interval_sec: float = 15 * 86400.0,
                              amplitude: float = 4.0):
    """Creates a deterministic source function for reproducible comparison.

    Returns a source function and its internal state dictionary.

    Args:
        cycle_period_yr: Sunspot cycle period in years.
        emergence_interval_sec: Time between source emergences.
        amplitude: Peak field amplitude per source.

    Returns:
        Tuple of (source_function, state_dict).
    """
    # Pre-compute all emergence events for reproducibility
    rng = np.random.default_rng(123)
    t_total_sec = cycle_period_yr * SECONDS_PER_YEAR
    events = []
    t = 0.0
    while t < t_total_sec:
        t_yr = t / SECONDS_PER_YEAR
        phase = t_yr / cycle_period_yr
        activity = np.sin(np.pi * phase)**2

        if activity > 0.05:
            # Spoerer's law
            lat_center = 30.0 - 20.0 * phase
            lat = lat_center + rng.normal(0, 2.0)
            tilt = 5.0 + 0.3 * abs(lat)
            amp = amplitude * activity
            events.append((t, abs(lat), amp, tilt))

        t += emergence_interval_sec

    state = {'event_idx': 0, 'events': events}

    def source_func(solver, t_sec):
        """Adds pre-computed source events."""
        idx = state['event_idx']
        while idx < len(events) and events[idx][0] <= t_sec:
            _, lat, amp, tilt = events[idx]
            solver.add_source(lat_deg=lat, width_deg=2.5,
                              amplitude=amp, tilt_deg=tilt)
            solver.add_source(lat_deg=-lat, width_deg=2.5,
                              amplitude=amp, tilt_deg=tilt)
            idx += 1
        state['event_idx'] = idx

    return source_func, state


# ── Run three cases ──
cases = {
    'No transport\n(소스만 / sources only)': {'eta': 0.01, 'v_max': 0.0},
    'Diffusion only\n(확산만)': {'eta': 600.0, 'v_max': 0.0},
    'Diffusion + Flow\n(확산 + 유동)': {'eta': 600.0, 'v_max': 11.0},
}

results_p3 = {}
for label, params in cases.items():
    solver = FluxTransportSolver(n_lat=361, **params)
    src_func, src_state = make_deterministic_source()
    result = solver.run(t_years=11.0, snapshot_interval_years=0.1,
                        source_func=src_func)
    results_p3[label] = result
    print(f"  Completed: {label.split(chr(10))[0]}")

# ── Plot butterfly diagrams ──
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True, sharey=True)

for ax, (label, result) in zip(axes, results_p3.items()):
    T, L = np.meshgrid(result['times'], result['lats_deg'])
    # Use consistent colorscale across all panels
    vmax = np.percentile(np.abs(result['Br_snapshots']), 97)
    vmax = max(vmax, 0.5)

    im = ax.pcolormesh(T, L, result['Br_snapshots'].T,
                       cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                       shading='auto')
    ax.set_ylabel('Latitude [deg]\n위도 [도]')
    ax.set_title(label, fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax, label='$B_r$ [G]', shrink=0.8)

axes[-1].set_xlabel('Time [years] / 시간 [년]')
plt.suptitle('세 가지 수송 체제 비교 (cf. Sheeley 2005, Figure 3)\n'
             'Three Transport Regimes Comparison',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("\n=== Part 3: 핵심 관찰 / Key Observations ===")
print("  1. 수송 없음: 소스가 활동대에 누적될 뿐, 극으로 확산되지 않음")
print("     No transport: sources accumulate in activity belts only")
print("  2. 확산만: 자기장이 느리게 퍼지며 반대 극성이 상쇄됨")
print("     Diffusion only: field spreads slowly, opposite polarities cancel")
print("  3. 확산+유동: 극 방향 surge가 형성되어 뚜렷한 줄무늬 패턴 생성")
print("     Diffusion+flow: poleward surges form distinct streak patterns")

---
## Part 4: 매개변수 민감도 연구 / Parameter Sensitivity Study

### 논문의 핵심 통찰 / Key Insight from the Paper

Sheeley (2005)는 flux-transport 모델의 매개변수에 대한 중요한 통찰을 제공합니다:

Sheeley (2005) provides crucial insight about the model parameters:

- **확산 계수** $\eta$가 극 자기장 강도를 결정하는 핵심 매개변수입니다.
- **경도 방향 유동 속도** $v$가 빠를수록 극 자기장이 **약해집니다** — 이는 직관에 반하는 결과입니다!
  - 이유: 빠른 유동은 선행/후행 극성을 분리하기 전에 함께 극으로 쓸어가 상쇄시킵니다.
  - 느린 유동은 확산이 선행 극성을 적도에서 상쇄시킬 시간을 주어, 불균형 자속이 더 많이 극에 도달합니다.

- The **diffusion coefficient** $\eta$ is the key parameter controlling polar field strength.
- **Faster meridional flow** $v$ produces **weaker** polar fields — a counterintuitive result!
  - Reason: fast flow sweeps both leading/trailing polarities poleward together before diffusion can cancel the leading polarity at the equator.
  - Slow flow gives diffusion time to cancel leading polarity at the equator, so more unbalanced flux reaches the poles.

In [ ]:
"""Part 4: Parameter sensitivity — diffusion rate and flow speed."""


def run_sensitivity(eta: float, v_max: float,
                    t_years: float = 11.0) -> dict:
    """Runs a single cycle simulation and returns polar field metrics.

    Args:
        eta: Diffusion coefficient [km^2/s].
        v_max: Peak meridional flow speed [m/s].
        t_years: Simulation duration in years.

    Returns:
        Dictionary with polar field time series and final values.
    """
    solver = FluxTransportSolver(n_lat=361, eta=eta, v_max=v_max)

    # Initial polar field
    polar_w = np.sin(np.radians(15.0))
    solver.Br += 2.0 * np.exp(-0.5 * ((solver.x - 1.0) / polar_w)**2)
    solver.Br -= 2.0 * np.exp(-0.5 * ((solver.x + 1.0) / polar_w)**2)

    src_func, _ = make_deterministic_source(amplitude=4.0)
    result = solver.run(t_years=t_years, snapshot_interval_years=0.25,
                        source_func=src_func)

    north_idx = np.where(result['lats_deg'] > 75)[0]
    Br_north = np.mean(result['Br_snapshots'][:, north_idx], axis=1)

    return {
        'times': result['times'],
        'Br_north': Br_north,
        'Br_north_final': Br_north[-1],
        'Br_north_min': np.min(Br_north),
    }


# ── Scan over diffusion rates ──
eta_values = [200, 400, 600, 800, 1000]
v_fixed = 11.0

eta_results = {}
for eta in eta_values:
    eta_results[eta] = run_sensitivity(eta=eta, v_max=v_fixed)
    print(f"  eta={eta} km^2/s: final polar B = {eta_results[eta]['Br_north_final']:.3f} G")

# ── Scan over flow speeds ──
v_values = [0, 5, 8, 11, 15, 20, 30]
eta_fixed = 600.0

v_results = {}
for v in v_values:
    v_results[v] = run_sensitivity(eta=eta_fixed, v_max=v)
    print(f"  v={v} m/s: final polar B = {v_results[v]['Br_north_final']:.3f} G")

# ── Plot results ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top-left: Polar field vs time for different eta
ax = axes[0, 0]
for eta in eta_values:
    r = eta_results[eta]
    ax.plot(r['times'], r['Br_north'],
            label=f'$\\eta$ = {eta} km²/s', alpha=0.8)
ax.axhline(0, color='gray', lw=0.5)
ax.set_xlabel('Time [years] / 시간 [년]')
ax.set_ylabel('$\\langle B_r \\rangle_{N pole}$ [G]')
ax.set_title(f'확산 계수 변화 (v={v_fixed} m/s 고정)\n'
             f'Varying Diffusion Rate (v={v_fixed} m/s fixed)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Top-right: Polar field vs time for different v
ax = axes[0, 1]
for v in v_values:
    r = v_results[v]
    ax.plot(r['times'], r['Br_north'],
            label=f'v = {v} m/s', alpha=0.8)
ax.axhline(0, color='gray', lw=0.5)
ax.set_xlabel('Time [years] / 시간 [년]')
ax.set_ylabel('$\\langle B_r \\rangle_{N pole}$ [G]')
ax.set_title(f'유동 속도 변화 ($\\eta$={eta_fixed:.0f} km²/s 고정)\n'
             f'Varying Flow Speed ($\\eta$={eta_fixed:.0f} km²/s fixed)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Bottom-left: Final polar field vs eta
ax = axes[1, 0]
etas = list(eta_results.keys())
finals = [eta_results[e]['Br_north_final'] for e in etas]
ax.plot(etas, finals, 'bo-', markersize=8, lw=2)
ax.set_xlabel('$\\eta$ [km²/s]')
ax.set_ylabel('Final $\\langle B_r \\rangle_{N pole}$ [G]')
ax.set_title('최종 극 자기장 vs 확산 계수\nFinal Polar Field vs Diffusion Rate')
ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', lw=0.5)

# Bottom-right: Final polar field vs v
ax = axes[1, 1]
vs = list(v_results.keys())
finals_v = [v_results[v]['Br_north_final'] for v in vs]
ax.plot(vs, finals_v, 'rs-', markersize=8, lw=2)
ax.set_xlabel('$v_{max}$ [m/s]')
ax.set_ylabel('Final $\\langle B_r \\rangle_{N pole}$ [G]')
ax.set_title('최종 극 자기장 vs 유동 속도\nFinal Polar Field vs Flow Speed')
ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', lw=0.5)

# Annotate the counterintuitive result
ax.annotate('빠른 유동 → 약한 극 자기장\nFaster flow → weaker polar field',
            xy=(20, finals_v[vs.index(20)]),
            xytext=(22, finals_v[vs.index(20)] + 0.5),
            fontsize=9, color='red',
            arrowprops=dict(arrowstyle='->', color='red'))

plt.tight_layout()
plt.show()

print("\n=== Part 4: 핵심 결과 / Key Results ===")
print("  확산 계수가 클수록 더 많은 상쇄 → 더 약한 극 자기장")
print("  Higher diffusion → more cancellation → weaker polar field")
print("  빠른 유동 → 상쇄 전에 쌍극을 극으로 수송 → 더 약한 극 자기장")
print("  Faster flow → transports doublets before cancellation → weaker polar field")

---
## Part 5: 다중 주기 시뮬레이션과 가변 유동 / Multi-Cycle Simulation with Variable Flow

### 배경 / Background

Sheeley (2005)는 경도 방향 유동 속도가 태양 주기에 따라 변할 수 있음을 논의합니다.
관측에 의하면:

Sheeley (2005) discusses how meridional flow speed may vary with the solar cycle.
Observations suggest:

- **활발한 주기** → **빠른 유동** (약 $v + \Delta v$)
- **조용한 주기** → **느린 유동** (약 $v - \Delta v$)

- **Active cycle** → **faster flow** ($\sim v + \Delta v$)
- **Quiet cycle** → **slower flow** ($\sim v - \Delta v$)

이 관계는 자기 feedback을 통해 이전 주기의 활동 수준이 다음 주기의 극 자기장 강도에 영향을 미칩니다.

This relationship creates a magnetic feedback where the activity level of one cycle influences
the polar field strength of the next cycle.

### 시뮬레이션 설계 / Simulation Design

- 3개의 연속적인 11년 주기를 시뮬레이션합니다.
- 각 주기의 유동 속도를 $11 \pm 6$ m/s 범위에서 변화시킵니다.
- 극 자기장 역전이 각 주기마다 보존되는지 확인합니다.

- Simulate 3 consecutive 11-year cycles.
- Vary flow speed in the range $11 \pm 6$ m/s for each cycle.
- Verify that polar field reversal is preserved in each cycle.

In [ ]:
"""Part 5: Multi-cycle simulation with variable meridional flow."""


def make_multicycle_source(n_cycles: int = 3,
                           cycle_period_yr: float = 11.0,
                           amplitude_per_cycle: list = None):
    """Creates source events spanning multiple solar cycles.

    Each cycle can have a different amplitude (activity level).
    Hale's law polarity alternates between odd and even cycles.

    Args:
        n_cycles: Number of sunspot cycles.
        cycle_period_yr: Period of each cycle in years.
        amplitude_per_cycle: List of amplitude scaling factors per cycle.

    Returns:
        Tuple of (source_function, events_list, state_dict).
    """
    if amplitude_per_cycle is None:
        amplitude_per_cycle = [4.0] * n_cycles

    rng = np.random.default_rng(456)
    events = []
    emergence_interval = 15 * 86400.0  # 15 days

    for cycle_idx in range(n_cycles):
        t_offset = cycle_idx * cycle_period_yr * SECONDS_PER_YEAR
        t_cycle_total = cycle_period_yr * SECONDS_PER_YEAR
        amp_scale = amplitude_per_cycle[cycle_idx]
        # Hale's law: polarity reverses every cycle
        polarity = 1.0 if cycle_idx % 2 == 0 else -1.0

        t = 0.0
        while t < t_cycle_total:
            phase = t / t_cycle_total
            activity = np.sin(np.pi * phase)**2

            if activity > 0.05:
                lat_center = 30.0 - 20.0 * phase
                lat = abs(lat_center + rng.normal(0, 2.0))
                tilt = 5.0 + 0.3 * lat
                amp = amp_scale * activity
                events.append((t + t_offset, lat, amp, tilt, polarity))

            t += emergence_interval

    state = {'event_idx': 0}

    def source_func(solver, t_sec):
        """Adds pre-computed multicycle source events."""
        idx = state['event_idx']
        while idx < len(events) and events[idx][0] <= t_sec:
            _, lat, amp, tilt, pol = events[idx]
            # Polarity determines which hemisphere gets which sign
            solver.add_source(lat_deg=lat, width_deg=2.5,
                              amplitude=amp * pol, tilt_deg=tilt)
            solver.add_source(lat_deg=-lat, width_deg=2.5,
                              amplitude=amp * pol, tilt_deg=tilt)
            idx += 1
        state['event_idx'] = idx

    return source_func, events, state


# ── Define cycle parameters ──
n_cycles = 3
cycle_yr = 11.0
total_years = n_cycles * cycle_yr

# Variable flow speeds: active → fast, quiet → slow
cycle_flow_speeds = [14.0, 8.0, 17.0]  # m/s per cycle
cycle_amplitudes = [5.0, 3.0, 6.0]     # Correlated: active→fast flow

print("=== 다중 주기 시뮬레이션 설정 / Multi-Cycle Setup ===")
for i, (v, a) in enumerate(zip(cycle_flow_speeds, cycle_amplitudes)):
    activity_label = "활발 / Active" if a > 4 else "조용 / Quiet"
    print(f"  Cycle {i+1}: v = {v} m/s, amplitude = {a} G ({activity_label})")

# ── Run simulation with time-varying flow ──
solver = FluxTransportSolver(n_lat=361, eta=600.0, v_max=cycle_flow_speeds[0])

# Initial polar field
polar_w = np.sin(np.radians(15.0))
solver.Br += 2.0 * np.exp(-0.5 * ((solver.x - 1.0) / polar_w)**2)
solver.Br -= 2.0 * np.exp(-0.5 * ((solver.x + 1.0) / polar_w)**2)

src_func, events, src_state = make_multicycle_source(
    n_cycles=n_cycles, amplitude_per_cycle=cycle_amplitudes)

# We need to manually handle time-varying flow speed
dt = solver.compute_dt()
t_total_sec = total_years * SECONDS_PER_YEAR
snapshot_interval = 0.2 * SECONDS_PER_YEAR

times_all = []
snapshots_all = []
t = 0.0
next_snapshot = 0.0

while t < t_total_sec:
    # Update flow speed based on current cycle
    current_cycle = min(int(t / (cycle_yr * SECONDS_PER_YEAR)), n_cycles - 1)
    solver.v_max = cycle_flow_speeds[current_cycle]

    # Snapshot
    if t >= next_snapshot:
        times_all.append(t / SECONDS_PER_YEAR)
        snapshots_all.append(solver.Br.copy())
        next_snapshot += snapshot_interval

    # Sources
    src_func(solver, t)

    # Step
    actual_dt = min(dt, t_total_sec - t)
    solver.step(actual_dt)
    t += actual_dt

times_all.append(t / SECONDS_PER_YEAR)
snapshots_all.append(solver.Br.copy())
times_all = np.array(times_all)
snapshots_all = np.array(snapshots_all)

print(f"\n  시뮬레이션 완료 / Simulation complete: {len(times_all)} snapshots over {total_years} years")

# ── Plot results ──
fig = plt.figure(figsize=(15, 10))
gs = GridSpec(3, 1, height_ratios=[2, 1, 1], hspace=0.35)

# Top: Butterfly diagram
ax = fig.add_subplot(gs[0])
T, L = np.meshgrid(times_all, solver.lats_deg)
vmax = np.percentile(np.abs(snapshots_all), 97)
im = ax.pcolormesh(T, L, snapshots_all.T,
                   cmap='RdBu_r', vmin=-vmax, vmax=vmax, shading='auto')
plt.colorbar(im, ax=ax, label='$B_r$ [G]', shrink=0.8)
ax.set_ylabel('Latitude [deg] / 위도 [도]')
ax.set_title('3주기 Flux-Transport 시뮬레이션 (가변 유동)\n'
             'Three-Cycle Flux Transport (Variable Flow)')

# Mark cycle boundaries
for i in range(1, n_cycles):
    ax.axvline(i * cycle_yr, color='white', ls='--', lw=1.5, alpha=0.7)
    ax.text(i * cycle_yr + 0.3, 80, f'Cycle {i+1}', color='white', fontsize=9)
ax.text(0.3, 80, 'Cycle 1', color='white', fontsize=9)

# Middle: Polar field
ax = fig.add_subplot(gs[1])
north_idx = np.where(solver.lats_deg > 75)[0]
south_idx = np.where(solver.lats_deg < -75)[0]
Br_north = np.mean(snapshots_all[:, north_idx], axis=1)
Br_south = np.mean(snapshots_all[:, south_idx], axis=1)

ax.plot(times_all, Br_north, 'r-', label='North / 북극', lw=2)
ax.plot(times_all, Br_south, 'b-', label='South / 남극', lw=2)
ax.axhline(0, color='gray', lw=0.5)
for i in range(1, n_cycles):
    ax.axvline(i * cycle_yr, color='gray', ls='--', lw=1, alpha=0.5)
ax.set_ylabel('$\\langle B_r \\rangle_{polar}$ [G]')
ax.set_title('극 자기장 진화 / Polar Field Evolution')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Bottom: Flow speed timeline
ax = fig.add_subplot(gs[2])
t_flow = np.linspace(0, total_years, 1000)
v_flow = np.array([cycle_flow_speeds[min(int(t / cycle_yr), n_cycles - 1)] for t in t_flow])
ax.fill_between(t_flow, v_flow, alpha=0.3, color='green')
ax.plot(t_flow, v_flow, 'g-', lw=2)
for i in range(1, n_cycles):
    ax.axvline(i * cycle_yr, color='gray', ls='--', lw=1, alpha=0.5)
ax.set_xlabel('Time [years] / 시간 [년]')
ax.set_ylabel('$v_{max}$ [m/s]')
ax.set_title('경도 방향 유동 속도 / Meridional Flow Speed')
ax.set_ylim(0, 25)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Summary ──
print("\n=== Part 5: 다중 주기 결과 요약 / Multi-Cycle Results ===")
for i in range(n_cycles):
    t_start = i * cycle_yr
    t_end = (i + 1) * cycle_yr
    mask = (times_all >= t_start) & (times_all <= t_end)
    Bn = Br_north[mask]
    print(f"  Cycle {i+1}: v={cycle_flow_speeds[i]} m/s, amp={cycle_amplitudes[i]} G")
    print(f"    N극 최소/최대: {Bn.min():.3f} / {Bn.max():.3f} G")
    print(f"    N polar min/max: {Bn.min():.3f} / {Bn.max():.3f} G")

print("\n  핵심 결과: 극 자기장 역전은 가변 유동에도 불구하고 매 주기 발생합니다.")
print("  Key result: Polar field reversal occurs each cycle despite variable flow.")

---
## Part 6: 요약 표 / Summary Table

### 파트별 연결 / Part-by-Part Connections

아래 표는 각 파트가 Sheeley (2005) 논문의 어떤 부분과 연결되는지, 그리고 다음 논문과의 관계를 요약합니다.

The table below summarizes how each part connects to Sheeley (2005) and the broader context.

In [ ]:
"""Part 6: Summary table connecting implementation to paper and context."""

print("=" * 90)
print("  Part 6: 요약 표 / Summary Table")
print("=" * 90)

headers = ["Part", "구현 내용 / Implementation", "논문 연결 / Paper Section",
           "다음 논문 / Next Paper"]

rows = [
    ["Part 1",
     "1D 축대칭 FT 솔버\n1D axisymmetric FT solver",
     "Eq. 1, Sections 2-3\n(flux-transport equation)",
     "Wang & Sheeley (1991)\n원 논문 / original paper"],

    ["Part 2",
     "Joy's law doublet 및 극 역전\nJoy's law & polar reversal",
     "Section 4\n(Joy's law tilt discussion)",
     "Wang et al. (2002)\n극 자기장 예측 / polar field"],

    ["Part 3",
     "3가지 수송 체제 비교\nThree transport regimes",
     "Figure 3\n(key comparison figure)",
     "Leighton (1964)\n확산만 모델 / diffusion-only"],

    ["Part 4",
     "eta, v 매개변수 민감도\nParameter sensitivity",
     "Section 5\n(parameter discussion)",
     "Baumann et al. (2004)\n최적 매개변수 / optimal params"],

    ["Part 5",
     "가변 유동 다중 주기\nMulti-cycle variable flow",
     "Section 6\n(variable flow discussion)",
     "Schrijver & Liu (2008)\n주기 예측 / cycle prediction"],
]

# Print formatted table
for header in headers:
    print(f"  {header}")
print("-" * 90)
for row in rows:
    print(f"\n  [{row[0]}]")
    for line in row[1].split('\n'):
        print(f"    Implementation: {line}")
    for line in row[2].split('\n'):
        print(f"    Paper:          {line}")
    for line in row[3].split('\n'):
        print(f"    Next:           {line}")

print("\n" + "=" * 90)
print("  핵심 교훈 / Key Lessons")
print("=" * 90)
lessons = [
    ("Flux-transport 방정식은 확산 + 유동 + 소스의 세 항으로 구성됩니다.",
     "The FT equation has three terms: diffusion + meridional flow + sources."),
    ("Joy's law 기울기가 극 자기장 역전의 근본 원인입니다.",
     "Joy's law tilt is the fundamental cause of polar field reversal."),
    ("빠른 유동은 직관과 달리 더 약한 극 자기장을 만듭니다.",
     "Faster flow counterintuitively produces weaker polar fields."),
    ("이 모델(SFT)은 현대 태양 주기 예측의 기초입니다.",
     "This model (SFT) is the foundation of modern solar cycle prediction."),
]
for i, (kr, en) in enumerate(lessons, 1):
    print(f"\n  {i}. {kr}")
    print(f"     {en}")

print("\n" + "=" * 90)